# Bagh-Chal Deep Reinforcement Learning: Full Kaggle GPU Training Suite

**Target Venue**: Springer Nature (*Neural Computing & Applications*, *Applied Intelligence*)

This notebook provides the complete, publication-grade training suite for training Graph Neural Networks (GNN), 2D CNNs, and Flat MLPs on Bagh-Chal with:
- **Potential-Based Reward Shaping (PBRS)** (Ng et al., 1999)
- **Role-Decoupled Dual-Networks** vs Shared Single-Trunk Models
- **Standardized Periodic Evaluation** against Heuristic and Minimax Baselines (Wilson 95% Confidence Intervals)
- **Automatic Springer Nature Publication Plots** (300 DPI, Vector PDF & PNG)
- **Complete Checkpointing and Metric Logging**

### Step 1: Environment Variables & Hardware Verification
Verify GPU availability (NVIDIA Tesla T4 or P100) and configure optimal multi-threading and CUDA allocation.

In [ ]:
import os
import sys
import torch

# Configure environment variables
os.environ["BAGH_CHAL_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONPATH"] = "."
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["WANDB_MODE"] = "disabled"

sys.path.insert(0, os.path.abspath("."))

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU!")

### Step 2: Run PyTest Suite (65 Unit & Integration Tests)
Ensures all board rules, automorphisms, action masking, search algorithms, and neural networks pass.

In [ ]:
!python -m pytest tests/ -v

### Step 3: Fast Sanity Smoke-Test (30 Episodes)
Executes a quick 30-episode sanity run with live evaluation against heuristic and Minimax baselines.

In [ ]:
!python train_kaggle.py --smoke-test --representation gnn --architecture dual --reward-scheme pbrs

### Step 4: Full High-Scale Training (Flagship GNN Dual-Net: 500 Episodes)
Trains the flagship Graph Neural Network with role-specialized dual weights and PBRS potential reward shaping on the Kaggle GPU.

In [ ]:
!python train_kaggle.py \
    --representation gnn \
    --architecture dual \
    --reward-scheme pbrs \
    --episodes 500 \
    --rollout-size 25 \
    --ppo-epochs 4 \
    --batch-size 64 \
    --lr 0.0003 \
    --eval-interval 50 \
    --output-dir data/kaggle_run

### Step 5: Multi-Architecture Ablation Campaign (Optional for Paper)
Runs the 4-condition cross-architectural ablation study:
1. `GNN-Dual (Ours)`
2. `GNN-Single`
3. `CNN-Single`
4. `Flat-Single`
Generates unified comparative bar charts and LaTeX-ready summary tables.

In [ ]:
# Uncomment to execute the full multi-architecture ablation campaign:
# !python train_kaggle.py --mode ablation --episodes 200 --output-dir data/kaggle_run/ablation

### Step 6: Display Generated Publication Figures Directly in Notebook
Render the 300 DPI Springer Nature dashboard figures generated by the training run.

In [ ]:
from IPython.display import Image, display
import glob

png_plots = glob.glob("data/kaggle_run/**/figures/*.png", recursive=True)
if png_plots:
    for p in png_plots:
        print(f"Displaying: {p}")
        display(Image(filename=p))
else:
    print("No plot PNGs found yet. Run Step 4 or Step 5 first.")

### Step 7: Package and Download Artifacts
Zips all checkpoints (`best_model.pt`, `final_model.pt`), CSV history, JSON metrics, and vector PDFs into `kaggle_artifacts.zip` for one-click download from Kaggle's output pane.

In [ ]:
from train_kaggle import package_kaggle_artifacts
package_kaggle_artifacts(root_dir="data/kaggle_run", archive_name="kaggle_training_artifacts.zip")
!ls -lh kaggle_training_artifacts.zip